# 墓葬考古数据抽取与量化分析管线

目标：把考古简报/报告 PDF 自动转化为结构化数据库，支持类型学分析、年代-族属交叉分析和 QGIS 可视化。

流程：
1. PDF → Markdown（MinerU / marker-pdf）
2. Markdown → 结构化 JSON（LLM）
3. JSON → SQLite 数据库
4. 量化分析（交叉表、关联规则）
5. 导出 CSV/Shapefile 进 QGIS

## 1. 环境配置

In [ ]:
import os
import json
import sqlite3
import glob
import re
from pathlib import Path
from typing import List, Dict, Any

import pandas as pd

# 项目路径
PROJECT_DIR = Path("/root/xiaohanchen/projects/tomb-archaeology-pipeline")
PDF_DIR = PROJECT_DIR / "pdfs"
MARKDOWN_DIR = PROJECT_DIR / "markdown"
JSON_DIR = PROJECT_DIR / "json"
DB_PATH = PROJECT_DIR / "tombs.db"
EXPORT_DIR = PROJECT_DIR / "exports"

for d in [PDF_DIR, MARKDOWN_DIR, JSON_DIR, EXPORT_DIR]:
    d.mkdir(exist_ok=True)

# API 配置（从环境变量读取，避免硬编码）
KIMI_API_KEY = os.environ.get("KIMI_API_KEY", "")
KIMI_BASE_URL = os.environ.get("KIMI_BASE_URL", "https://api.kimi.com/coding/v1")

print(f"项目目录: {PROJECT_DIR}")
print(f"PDF 目录: {PDF_DIR}")
print(f"API Key 已配置: {bool(KIMI_API_KEY)}")

## 2. PDF → Markdown

这里提供两种方案：
- **方案 A**：使用 MinerU Cloud API（需要 MINERU_API_TOKEN）
- **方案 B**：本地 marker-pdf（需要 `pip install marker-pdf`）

先把所有 PDF 转成 Markdown，保留原始结构和图片占位符。

In [ ]:
def list_pdfs(pdf_dir: Path) -> List[Path]:
    """列出目录下所有 PDF 文件。"""
    return sorted(pdf_dir.glob("*.pdf"))

def pdf_to_markdown_mineru(pdf_path: Path, output_dir: Path, api_token: str) -> Path:
    """调用 MinerU Cloud API 将 PDF 转为 Markdown。"""
    import requests
    
    url = "https://mineru.net/api/v1/extract"
    with open(pdf_path, "rb") as f:
        files = {"file": f}
        headers = {"Authorization": f"Bearer {api_token}"}
        resp = requests.post(url, files=files, headers=headers, timeout=300)
    resp.raise_for_status()
    data = resp.json()
    
    output_path = output_dir / f"{pdf_path.stem}.md"
    output_path.write_text(data.get("markdown", ""), encoding="utf-8")
    return output_path

def pdf_to_markdown_marker(pdf_path: Path, output_dir: Path) -> Path:
    """使用本地 marker-pdf 转换（需提前安装）。"""
    from marker.converters.pdf import PdfConverter
    from marker.models import create_model_dict
    
    converter = PdfConverter(artifact_dict=create_model_dict())
    markdown = converter(str(pdf_path))
    output_path = output_dir / f"{pdf_path.stem}.md"
    output_path.write_text(markdown, encoding="utf-8")
    return output_path

# 查看当前有哪些 PDF
pdfs = list_pdfs(PDF_DIR)
print(f"找到 {len(pdfs)} 个 PDF")
for p in pdfs[:10]:
    print(" -", p.name)

## 3. Markdown → 结构化 JSON（LLM 抽取）

调用 Kimi/DeepSeek API，从 Markdown 文本中抽取墓葬字段。

In [ ]:
你是一位中国墓葬考古研究助理。请阅读以下考古简报/报告的 Markdown 文本，提取每个墓葬的结构化信息。

输出要求：
1. 只返回 JSON 数组，不要 Markdown 格式说明。
2. 数组中每个元素代表一个墓葬；必须提取文本中提到的所有墓葬，不要只返回第一个。
3. 如果某字段原文未提及，使用 null 或 "未提及"，不要编造。
3. 如果某字段原文未提及，使用 null 或 "未提及"，不要编造。
4. 年代和族属判断必须区分：是报告作者的结论，还是你自己的推断。
5. 如果图文冲突，优先按文字描述记录，并在 uncertainty 中标注。
6. 墓葬形制（tomb_form）必须给出单标签主分类，并从下面候选中挑选最贴切的一个；若原文有复合描述，把次要信息放进 tomb_type 字段。

候选形制单标签：
土坑竖穴墓 | 土坑洞室墓 | 砖室墓 | 砖椁墓 | 石室墓 | 土坑竖穴砖椁墓 | 舟形墓 | 瓮棺墓 | 其他

实测记录层字段（每个墓葬一条）：
- tomb_id: 墓葬编号
- site_name: 遗址/墓地名称
- location: 地理位置（省/市/县）
- tomb_form: 墓葬形制主标签（单标签，必须从候选形制中选择）
- tomb_form_confidence: 形制判断置信度（高/中/低）
- tomb_form_basis: 形制判断依据（文字描述/平面图/剖面图/图文一致/图文冲突）
- tomb_type: 报告原文对墓葬类型的称呼或复合形制描述
- orientation: 墓葬方向
- passage: 墓道有无及方向
- chamber_shape: 墓室平面形状（长方形/梯形/方形/圆形/椭圆形/舟形/不规则/未提及）
- chamber_size: 墓室尺寸（长×宽×深，厘米）
- sealing: 封门方式（无/夯土/砖封门/石封门/木板/未提及）
- coffin: 棺椁情况
- platform: 二层台有无（无/有（生土）/有（熟土）/未提及）
- niche: 壁龛有无（头龛/足龛/侧龛，数量）
- flooring: 铺地材料（无/青砖/石板/夯土/白灰/未提及）
- burial_position: 葬式（直肢/屈肢/二次葬/扰乱葬/仰身直肢/侧身屈肢/俯身/未提及）
- head_direction: 头向
- bone_preservation: 人骨保存情况（完好/较差/仅存牙齿/不存/未提及）
- sacrificial_animals: 殉葬动物种类及位置
- artifact_distribution: 随葬品空间分布
- artifact_types: 器类列表
- artifact_count: 随葬品总数
- artifact_combination: 组合描述
- special_artifacts: 特殊器物（铜鍑/带扣/马具等）
- reported_date: 报告定年
- dated_material: 纪年材料有无
- dating_basis: 定年依据
- special_customs: 特殊葬俗
- reported_ethnicity: 报告族属判断

证据层字段 evidence_records（数组）：
每条证据包含：
- interpretation_field: 支撑哪个字段的推断（tomb_form/chronology/social_rank/wealth/ethnicity/burial_custom/artifact_combination/other）
- evidence_type: 证据类型（coin_type/artifact_typology/tomb_structure/regional_comparison/stratigraphy/report_citation/text_description/image_plan/image_section/radiocarbon/dna/historical_record/other）
- evidence_content: 证据内容摘要
- report_citation: 报告中的具体引文、页码或图号
- evidence_confidence: 证据可信度（高/中/低/不确定）

不确定性层字段 uncertainty_records（数组）：
每条不确定性包含：
- field_name: 哪个字段存在不确定性
- confidence_score: 0-1 之间的置信度分数
- competing_interpretations: 不同解释或冲突点
- reviewer_notes: 需要人工复核的备注
- review_status: pending/confirmed/disputed/needs_review

注意：证据层和不确定性层如果原文没有明确冲突或特殊证据，可以为空数组，但不要省略字段。


def extract_with_llm(markdown_text: str, api_key: str, base_url: str, model: str = "kimi-for-coding") -> List[Dict[str, Any]]:
    """调用 Kimi API 从 Markdown 抽取墓葬数据。"""
    import requests
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    messages = [
        {"role": "system", "content": EXTRACTION_PROMPT},
        {"role": "user", "content": f"请提取以下文本中的墓葬信息：\n\n{markdown_text[:60000]}"}
    ]
    
    payload = {
        "model": model,
        "messages": messages,
        "response_format": {"type": "json_object"}
    }
    
    resp = requests.post(f"{base_url}/chat/completions", headers=headers, json=payload, timeout=300)
    resp.raise_for_status()
    content = resp.json()["choices"][0]["message"]["content"]
    
    try:
        parsed = json.loads(content)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            # 模型可能用不同键包装数组；优先返回第一个长度大于1的数组
            list_keys = [k for k, v in parsed.items() if isinstance(v, list)]
            if list_keys:
                # 优先选择看起来像墓葬数组的键
                for key in ["tombs", "result", "results", "data", "墓葬", "输出", "extracted_tombs"]:
                    if key in list_keys and len(parsed[key]) > 0:
                        return parsed[key]
                return parsed[list_keys[0]]
            # 单个对象
            return [parsed]
        return []
    except json.JSONDecodeError:
        print("LLM 返回非 JSON，尝试提取数组...")
        match = re.search(r"\[.*\]", content, re.DOTALL)
        return json.loads(match.group(0)) if match else []

# 示例：抽取单个 Markdown 文件
# md_files = sorted(MARKDOWN_DIR.glob("*.md"))
# sample_md = md_files[0].read_text(encoding="utf-8")
# result = extract_with_llm(sample_md, KIMI_API_KEY, KIMI_BASE_URL)
# print(json.dumps(result[:2], ensure_ascii=False, indent=2))


In [ ]:
SINGLE_TOMB_PROMPT = """你是一位中国墓葬考古研究助理。请阅读以下单个墓葬的描述，提取结构化信息。\n\n输出要求：\n1. 只返回一个 JSON 对象，不要 Markdown 格式说明，不要 JSON 数组包装。\n2. 如果某字段原文未提及，使用 null 或 "未提及"，不要编造。\n3. evidence_records 的 interpretation_field 和 evidence_type 必须使用下面英文候选值；uncertainty_records 的 review_status 必须使用英文候选值。\n\n候选形制单标签：\n土坑竖穴墓 | 土坑洞室墓 | 砖室墓 | 砖椁墓 | 石室墓 | 土坑竖穴砖椁墓 | 舟形墓 | 瓮棺墓 | 其他\n\n字段：\n- tomb_id: 墓葬编号\n- site_name: 遗址/墓地名称\n- location: 地理位置\n- tomb_form: 墓葬形制主标签（单标签）\n- tomb_form_confidence: 形制判断置信度（高/中/低）\n- tomb_form_basis: 形制判断依据（文字描述/平面图/剖面图/图文一致/图文冲突）\n- tomb_type: 报告原文称谓或复合形制描述\n- orientation: 墓葬方向\n- passage: 墓道有无及方向\n- chamber_shape: 墓室平面形状（长方形/梯形/方形/圆形/椭圆形/舟形/不规则/未提及）\n- chamber_size: 墓室尺寸（长×宽×深，厘米）\n- sealing: 封门方式（无/夯土/砖封门/石封门/木板/未提及）\n- coffin: 棺椁情况\n- platform: 二层台有无（无/有（生土）/有（熟土）/未提及）\n- niche: 壁龛有无（头龛/足龛/侧龛，数量）\n- flooring: 铺地材料（无/青砖/石板/夯土/白灰/未提及）\n- burial_position: 葬式（直肢/屈肢/二次葬/扰乱葬/仰身直肢/侧身屈肢/俯身/未提及）\n- head_direction: 头向\n- bone_preservation: 人骨保存情况（完好/较差/仅存牙齿/不存/未提及）\n- sacrificial_animals: 殉葬动物种类及位置\n- artifact_distribution: 随葬品空间分布\n- artifact_types: 器类列表\n- artifact_count: 随葬品总数\n- artifact_combination: 组合描述\n- special_artifacts: 特殊器物（铜鍑/带扣/马具等）\n- reported_date: 报告定年\n- dated_material: 纪年材料有无\n- dating_basis: 定年依据\n- special_customs: 特殊葬俗\n- reported_ethnicity: 报告族属判断\n- evidence_records: 证据数组（每条含 interpretation_field, evidence_type, evidence_content, report_citation, evidence_confidence）\n- uncertainty_records: 不确定性数组（每条含 field_name, confidence_score, competing_interpretations, reviewer_notes, review_status）\n- note: 其他备注\n\nevidence_records 受控词表（必须英文）：\n- interpretation_field: tomb_form / chronology / social_rank / wealth / ethnicity / burial_custom / artifact_combination / other\n- evidence_type: coin_type / artifact_typology / tomb_structure / regional_comparison / stratigraphy / report_citation / text_description / image_plan / image_section / radiocarbon / dna / historical_record / other\n- evidence_confidence: high / medium / low / uncertain\n\nuncertainty_records 受控词表（必须英文）：\n- review_status: pending / confirmed / disputed / needs_review\n"""\n\ndef chunk_tombs(md_text: str) -> List[str]:\n    """按墓葬编号把报告切成单个墓葬段落。"""\n    pattern = re.compile(r"(?=M\\d+\\s*位于)")\n    chunks = pattern.split(md_text)\n    return [c.strip() for c in chunks if re.match(r"M\\d+\\s*位于", c.strip())]\n\ndef extract_single_tomb(markdown_text: str, api_key: str, base_url: str, model: str = "kimi-for-coding") -> Dict[str, Any]:\n    """调用 Kimi API 从单个墓葬描述抽取。"""\n    import requests\n    headers = {\n        "Authorization": f"Bearer {api_key}",\n        "Content-Type": "application/json"\n    }\n    messages = [\n        {"role": "system", "content": SINGLE_TOMB_PROMPT},\n        {"role": "user", "content": f"请提取以下墓葬信息：\\n\\n{markdown_text[:12000]}"}\n    ]\n    payload = {\n        "model": model,\n        "messages": messages,\n        "response_format": {"type": "json_object"}\n    }\n    resp = requests.post(f"{base_url}/chat/completions", headers=headers, json=payload, timeout=300)\n    resp.raise_for_status()\n    content = resp.json()["choices"][0]["message"]["content"]\n    try:\n        parsed = json.loads(content)\n        if isinstance(parsed, dict):\n            return parsed\n        if isinstance(parsed, list) and len(parsed) > 0:\n            return parsed[0]\n        return {"error": "empty response", "raw": content}\n    except json.JSONDecodeError:\n        return {"error": "json decode failed", "raw": content[:500]}\n\ndef process_markdown_chunked(md_path: Path, json_dir: Path, api_key: str, base_url: str,\n                              max_tombs: int = None, sleep_seconds: float = 0.5) -> Path:\n    """分块处理单个 Markdown 文件：每个墓葬一次 API 调用。"""\n    import time\n    md_text = md_path.read_text(encoding="utf-8")\n    chunks = chunk_tombs(md_text)\n    if max_tombs:\n        chunks = chunks[:max_tombs]\n    results = []\n    for i, chunk in enumerate(chunks, 1):\n        tomb_id = re.match(r"M\\d+", chunk).group(0)\n        print(f"[{i}/{len(chunks)}] 抽取 {tomb_id} ...")\n        result = extract_single_tomb(chunk, api_key, base_url)\n        result["source_md"] = md_path.name\n        result["source_pdf"] = md_path.stem + ".pdf"\n        results.append(result)\n        time.sleep(sleep_seconds)\n    json_path = json_dir / f"{md_path.stem}.json"\n    json_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(f"完成：{json_path}，共 {len(results)} 座墓葬")\n    return json_path\n\n# 示例：只抽取前 5 座墓葬\n# md_path = sorted(MARKDOWN_DIR.glob("*.md"))[0]\n# json_path = process_markdown_chunked(md_path, JSON_DIR, KIMI_API_KEY, KIMI_BASE_URL, max_tombs=5)\n

## 4. 批量处理所有 PDF/Markdown

In [ ]:
def process_all_pdfs(pdf_dir: Path, markdown_dir: Path, json_dir: Path,
                     api_key: str, base_url: str,
                     extractor: str = "marker") -> List[Path]:
    """批量处理：PDF → Markdown → JSON。"""
    results = []
    pdfs = list_pdfs(pdf_dir)
    
    for pdf_path in pdfs:
        md_path = markdown_dir / f"{pdf_path.stem}.md"
        json_path = json_dir / f"{pdf_path.stem}.json"
        
        # 如果已有 JSON，跳过
        if json_path.exists():
            print(f"跳过（已有 JSON）: {pdf_path.name}")
            results.append(json_path)
            continue
        
        # PDF → Markdown
        if not md_path.exists():
            print(f"转换 PDF: {pdf_path.name}")
            if extractor == "mineru":
                md_path = pdf_to_markdown_mineru(pdf_path, markdown_dir, api_token=os.environ.get("MINERU_API_TOKEN"))
            else:
                md_path = pdf_to_markdown_marker(pdf_path, markdown_dir)
        
        # Markdown → JSON
        print(f"抽取: {pdf_path.name}")
        md_text = md_path.read_text(encoding="utf-8")
        tombs = extract_with_llm(md_text, api_key, base_url)
        
        # 添加来源信息
        for tomb in tombs:
            tomb["source_pdf"] = pdf_path.name
            tomb["source_md"] = md_path.name
        
        json_path.write_text(json.dumps(tombs, ensure_ascii=False, indent=2), encoding="utf-8")
        results.append(json_path)
    
    return results

# 运行批量处理（取消注释后执行）
# json_paths = process_all_pdfs(PDF_DIR, MARKDOWN_DIR, JSON_DIR, KIMI_API_KEY, KIMI_BASE_URL, extractor="marker")
# print(f"完成 {len(json_paths)} 个文件")

## 5. JSON → SQLite 数据库

In [ ]:
def init_database(db_path: Path):
    """初始化五层 ontology 数据库。"""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # 1. 实测记录层
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS tombs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        tomb_id TEXT,
        site_name TEXT,
        location TEXT,
        tomb_form TEXT,
        tomb_form_confidence TEXT,
        tomb_form_basis TEXT,
        tomb_type TEXT,
        orientation TEXT,
        passage TEXT,
        chamber_shape TEXT,
        chamber_size TEXT,
        sealing TEXT,
        coffin TEXT,
        platform TEXT,
        niche TEXT,
        flooring TEXT,
        burial_position TEXT,
        head_direction TEXT,
        bone_preservation TEXT,
        sacrificial_animals TEXT,
        artifact_distribution TEXT,
        artifact_types TEXT,
        artifact_count TEXT,
        artifact_combination TEXT,
        special_artifacts TEXT,
        reported_date TEXT,
        dated_material TEXT,
        dating_basis TEXT,
        special_customs TEXT,
        reported_ethnicity TEXT,
        source_pdf TEXT,
        source_md TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    # 2. 图像标签层
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS image_labels (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        tomb_id TEXT,
        site_name TEXT,
        source_pdf TEXT,
        source_image_filename TEXT,
        image_type TEXT,
        image_quality TEXT,
        visual_tomb_form TEXT,
        visual_confidence REAL,
        image_label_source TEXT,
        annotated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    # 3. 证据层
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS evidence_records (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        tomb_id TEXT,
        site_name TEXT,
        interpretation_field TEXT,
        evidence_type TEXT,
        evidence_content TEXT,
        report_citation TEXT,
        evidence_confidence TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    # 4. 分析层
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS analysis_results (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        tomb_id TEXT,
        site_name TEXT,
        analysis_type TEXT,
        analysis_result TEXT,
        confidence TEXT,
        evidence_summary TEXT,
        analyst_model TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    # 5. 不确定性层
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS uncertainty_records (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        tomb_id TEXT,
        site_name TEXT,
        field_name TEXT,
        confidence_score REAL,
        competing_interpretations TEXT,
        reviewer_notes TEXT,
        review_status TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    ''')
    
    conn.commit()
    conn.close()


def load_json_to_db(json_dir: Path, db_path: Path):
    """把 JSON 文件导入 SQLite 主表、证据层、不确定性层。"""
    init_database(db_path)
    conn = sqlite3.connect(db_path)
    
    tomb_cols = [
        "tomb_id", "site_name", "location", "tomb_form", "tomb_form_confidence",
        "tomb_form_basis", "tomb_type", "orientation", "passage", "chamber_shape",
        "chamber_size", "sealing", "coffin", "platform", "niche", "flooring",
        "burial_position", "head_direction", "bone_preservation", "sacrificial_animals",
        "artifact_distribution", "artifact_types", "artifact_count",
        "artifact_combination", "special_artifacts", "reported_date",
        "dated_material", "dating_basis", "special_customs", "reported_ethnicity",
        "source_pdf", "source_md"
    ]
    
    for json_path in sorted(json_dir.glob("*.json")):
        tombs = json.loads(json_path.read_text(encoding="utf-8"))
        for tomb in tombs:
            # 主表
            values = [str(tomb.get(c, "")) if tomb.get(c, "") is not None else "" for c in tomb_cols]
            placeholders = ", ".join(["?"] * len(tomb_cols))
            conn.execute(f"INSERT INTO tombs ({', '.join(tomb_cols)}) VALUES ({placeholders})", values)
            
            # 证据层
            for ev in tomb.get("evidence_records", []):
                ev_cols = ["tomb_id", "site_name", "interpretation_field", "evidence_type", "evidence_content", "report_citation", "evidence_confidence"]
                ev_vals = [
                    tomb.get("tomb_id", ""),
                    tomb.get("site_name", ""),
                    ev.get("interpretation_field", ""),
                    ev.get("evidence_type", ""),
                    ev.get("evidence_content", ""),
                    ev.get("report_citation", ""),
                    ev.get("evidence_confidence", "")
                ]
                conn.execute(f"INSERT INTO evidence_records ({', '.join(ev_cols)}) VALUES ({', '.join(['?'] * len(ev_cols))})", ev_vals)
            
            # 不确定性层
            for un in tomb.get("uncertainty_records", []):
                un_cols = ["tomb_id", "site_name", "field_name", "confidence_score", "competing_interpretations", "reviewer_notes", "review_status"]
                un_vals = [
                    tomb.get("tomb_id", ""),
                    tomb.get("site_name", ""),
                    un.get("field_name", ""),
                    un.get("confidence_score", 0.5),
                    un.get("competing_interpretations", ""),
                    un.get("reviewer_notes", ""),
                    un.get("review_status", "pending")
                ]
                conn.execute(f"INSERT INTO uncertainty_records ({', '.join(un_cols)}) VALUES ({', '.join(['?'] * len(un_cols))})", un_vals)
    
    conn.commit()
    conn.close()


def insert_image_labels(db_path: Path, records: List[Dict[str, Any]]):
    """批量写入图像标签层。"""
    conn = sqlite3.connect(db_path)
    cols = ["tomb_id", "site_name", "source_pdf", "source_image_filename",
            "image_type", "image_quality", "visual_tomb_form", "visual_confidence",
            "image_label_source"]
    placeholders = ", ".join(["?"] * len(cols))
    for r in records:
        vals = [r.get(c, "") for c in cols]
        conn.execute(f"INSERT INTO image_labels ({', '.join(cols)}) VALUES ({placeholders})", vals)
    conn.commit()
    conn.close()


def insert_analysis_results(db_path: Path, records: List[Dict[str, Any]]):
    """批量写入分析层。"""
    conn = sqlite3.connect(db_path)
    cols = ["tomb_id", "site_name", "analysis_type", "analysis_result",
            "confidence", "evidence_summary", "analyst_model"]
    placeholders = ", ".join(["?"] * len(cols))
    for r in records:
        vals = [r.get(c, "") for c in cols]
        conn.execute(f"INSERT INTO analysis_results ({', '.join(cols)}) VALUES ({placeholders})", vals)
    conn.commit()
    conn.close()



## 6. 数据查看与清洗

In [ ]:
# 读取数据库
df = pd.read_sql("SELECT * FROM tombs", sqlite3.connect(DB_PATH))
print(f"总记录数: {len(df)}")
print(df.columns.tolist())
df.head()

In [ ]:
# 快速查看关键字段的分布
for col in ["tomb_type", "chamber_shape", "burial_position", "reported_date", "reported_ethnicity"]:
    print(f"\n=== {col} ===")
    print(df[col].value_counts().head(10))

## 7. 量化分析

### 7.1 交叉表：年代 × 族属 × 墓葬类型

In [ ]:
# 年代 × 族属 交叉表
pd.crosstab(df["reported_date"], df["reported_ethnicity"], margins=True)

In [ ]:
# 墓葬类型 × 年代 交叉表
pd.crosstab(df["tomb_type"], df["reported_date"], margins=True)

### 7.2 关联规则：发现器物组合规律

例如你提到的「足龛 + 两个陶壶」这类微妙模式。

In [ ]:
# 需要安装 mlxtend
# !pip install mlxtend

from mlxtend.frequent_patterns import apriori, association_rules

def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """把墓葬特征转为二元矩阵，用于关联规则。"""
    features = pd.DataFrame(index=df.index)
    
    # 墓葬类型
    features["tomb_type_土坑"] = df["tomb_type"].str.contains("土坑", na=False).astype(int)
    features["tomb_type_砖室"] = df["tomb_type"].str.contains("砖室", na=False).astype(int)
    features["tomb_type_洞室"] = df["tomb_type"].str.contains("洞室", na=False).astype(int)
    
    # 壁龛
    features["has_niche"] = (df["niche"].notna() & (df["niche"] != "无") & (df["niche"] != "未提及")).astype(int)
    features["foot_niche"] = df["niche"].str.contains("足龛", na=False).astype(int)
    features["head_niche"] = df["niche"].str.contains("头龛", na=False).astype(int)
    
    # 器物组合关键词
    features["pottery_jars"] = df["artifact_combination"].str.contains("陶壶", na=False).astype(int)
    features["two_pottery_jars"] = df["artifact_combination"].str.contains("陶壶.*2|两个陶壶", na=False).astype(int)
    features["has_bronze_cauldron"] = df["special_artifacts"].str.contains("铜鍑", na=False).astype(int)
    features["has_horse_gear"] = df["special_artifacts"].str.contains("马具", na=False).astype(int)
    
    # 葬式
    features["flexed"] = df["burial_position"].str.contains("屈肢", na=False).astype(int)
    features["extended"] = df["burial_position"].str.contains("直肢", na=False).astype(int)
    
    return features

features = build_feature_matrix(df)
frequent_itemsets = apriori(features, min_support=0.05, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

# 按 lift 排序查看最强关联
rules.sort_values("lift", ascending=False).head(20)[["antecedents", "consequents", "support", "confidence", "lift"]]

## 8. 导出给 QGIS

In [ ]:
# 导出完整数据为 CSV
csv_path = EXPORT_DIR / "tombs.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"CSV 导出: {csv_path}")

# 如果你有地理坐标，可以导出 GeoJSON/Shapefile
# 这里假设你后续会用高德/百度 geocoding 或 CHGIS 给 location 字段配坐标

# 示例：为有坐标的记录生成 GeoJSON
# import geopandas as gpd
# from shapely.geometry import Point
# df_geo = df[df["longitude"].notna()].copy()
# geometry = [Point(xy) for xy in zip(df_geo["longitude"], df_geo["latitude"])]
# gdf = gpd.GeoDataFrame(df_geo, geometry=geometry, crs="EPSG:4326")
# gdf.to_file(EXPORT_DIR / "tombs.geojson", driver="GeoJSON")

## 9. 人工复核清单

LLM 抽取后，建议按以下优先级人工复核：
1. **图文冲突项**：note 字段中标注了冲突的墓葬
2. **关键判断项**：reported_date、reported_ethnicity、artifact_combination
3. **稀有特征项**：special_artifacts、special_customs
4. **数量字段**：artifact_count、chamber_size 等是否带单位

## 10. Next steps

- 把真实 PDF 放入 `./pdfs/` 目录
- 配置环境变量 `KIMI_API_KEY` 和（可选）`MINERU_API_TOKEN`
- 运行第 2-5 步生成数据库
- 用第 7 步分析具体规律
- 给 location 字段配经纬度后，导入 QGIS 做空间分析